# 🌑 Apollo Landing Site — 1-D Lunar Thermal Model
## Presentation Notebook

A focused, publication-quality exploration of the lunar regolith thermal model,
validated against Apollo 15 and Apollo 17 Heat Flow Experiment data.

**Sections**
1. Diurnal Temperature Cycles (depth profiles + amplitude decay + polar clock)
2. Subsurface Temperature Heatmap (depth × time)
3. Analytical Surface Temperature Map (local terrain)
4. Apollo Sensor Stability Timelines
5. Apollo HFE Validation — TG / TR / TC sensor types
6. Discrete vs Hayne 2017 Model Comparison
7. Probe Diurnal Cycles vs Both Models
8. Borestem Fiberglass Thermal Correction + 2-D Field
9. Borestem k-Fiberglass Sensitivity Sweep
10. Animated GIF — T(z) Profile over One Lunar Day
11. Animated GIF — Heatmap + Live Profile Cursor

> **Run all cells top-to-bottom.** Cells 00–04 set up all data; subsequent cells are independent visualisations.

## 0. Imports
*Run this cell first.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os, time, warnings
%matplotlib inline

import importlib
from lunar import (constants, models, dem, horizon, solar,
                   solver, analysis, plots, hfe_loader, borestem, borestem2d)
for _m in (constants, models, dem, horizon, solar,
           solver, analysis, plots, hfe_loader, borestem, borestem2d):
    importlib.reload(_m)

from lunar.borestem2d import solve_borestem_2d_steady

# Ensure GIF output directory exists
os.makedirs('gifs', exist_ok=True)

# plots.py already sets publication-quality rcParams at import time.
# Bump DPI for crisp display and high-res save.
mpl.rcParams['figure.dpi']   = 130
mpl.rcParams['savefig.dpi']  = 300
mpl.rcParams['savefig.bbox'] = 'tight'

warnings.filterwarnings('ignore', category=UserWarning)
print('✓ All modules loaded')

## 1. Configuration
*Edit this cell to change the target location, model, or energy-balance parameters.*

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  SIMULATION CONFIGURATION — edit here                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ── Target location (used for §1–3; Apollo sites always run in §4–9) ────
LAT =  26.1323   # degrees N  — Apollo 15 by default
LON =   3.6285   # degrees E  (0–360)
# Apollo 15: LAT=26.1323, LON=3.6285
# Apollo 17: LAT=20.1911, LON=30.7723

# ── Density model ─────────────────────────────────────────────────────────
MODEL   = 'discrete'   # 'discrete' | 'hayne_exponential'
H_PARAM = 0.07          # scale height / Layer-1 thickness (m)

# ── Energy balance ────────────────────────────────────────────────────────
SUNSCALE   = 1.10   # solar flux multiplier
ALBEDO     = 0.09   # Bond albedo
EMISSIVITY = 0.95   # IR emissivity
CHI        = 2.7    # radiative conductivity exponent

# ── Simulation ────────────────────────────────────────────────────────────
NDAYS = 3   # lunar days (first N-1 = spin-up, last = analysis)

# ── Apollo reference coordinates (do not change) ──────────────────────────
APOLLO_COORDS = {
    'Apollo 15': {'lat': 26.1323, 'lon':  3.6285, 'T_surf_est': 250.0},
    'Apollo 17': {'lat': 20.1911, 'lon': 30.7723, 'T_surf_est': 252.0},
}

# ── Derived (do not edit) ─────────────────────────────────────────────────
MODEL_ID  = models.MODEL_ID_MAP[MODEL]
HAYNE_ID  = models.MODEL_ID_MAP['hayne_exponential']
DISC_ID   = models.MODEL_ID_MAP['discrete']
models.set_hayne_h(H_PARAM)
models.set_layer1_h(H_PARAM)

print(f'Target   : {LAT:.4f}°N, {LON:.4f}°E')
print(f'Model    : {MODEL}  (id={MODEL_ID})')
print(f'SUNSCALE : {SUNSCALE}   CHI : {CHI}   ALBEDO : {ALBEDO}')
print(f'NDAYS    : {NDAYS}  ({NDAYS*29.53:.1f} Earth days)')

## 2. Load DEM, Snap Target Point, Compute Horizon

The LOLA Digital Elevation Model (4 pix/deg, ~7.6 km/pixel) is loaded once and reused for all subsequent computations.

In [ ]:
# ── Load LOLA elevation grid ──────────────────────────────────────────────
ELEV_M, PIXEL_M, MAP_RES, _ = dem.load_ldem()

# ── Snap to DEM pixel and extract terrain properties ──────────────────────
(ROW, COL,
 ACTUAL_LAT, ACTUAL_LON,
 ELEVATION, SLOPE, ASPECT) = dem.extract_point(LAT, LON, ELEV_M, PIXEL_M, MAP_RES)

# ── Horizon profile (360 directions = 1° resolution) ─────────────────────
N_AZ      = 360
AZ_ANGLES = np.linspace(0, 2*np.pi, N_AZ, endpoint=False, dtype=np.float32)

print('Computing horizon profile …', end=' ', flush=True)
t0 = time.time()
HORIZONS = horizon.compute_horizon_profile(
    ROW, COL, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=3000)
SVF = horizon.compute_sky_view_factor(HORIZONS)
print(f'done in {time.time()-t0:.1f} s')

print(f'  DEM      : {ELEV_M.shape[0]}×{ELEV_M.shape[1]} px, {MAP_RES} pix/deg')
print(f'  Snapped  : {ACTUAL_LAT:.4f}°N, {ACTUAL_LON:.4f}°E')
print(f'  Elevation: {ELEVATION:.0f} m')
print(f'  Slope    : {np.degrees(SLOPE):.2f}°   Aspect: {np.degrees(ASPECT):.1f}°')
print(f'  SVF      : {SVF:.3f}   Max horizon: {np.degrees(HORIZONS.max()):.1f}°')

## 3. Run Thermal Model at Target Site

The 1-D finite-difference solver integrates the heat-conduction equation
ρ(z)·c(T)·∂T/∂t = ∂/∂z[k(T,z)·∂T/∂z] over **NDAYS** lunar days.
The first (NDAYS−1) days act as spin-up; analysis uses the final day only.

In [ ]:
# ── Depth grid ────────────────────────────────────────────────────────────
Z_GRID = solver.create_depth_grid()
print(f'Depth grid: {len(Z_GRID)} nodes, 0 – {Z_GRID[-1]:.1f} m')

# ── Equilibrium initial condition (better deep temperatures than uniform T) 
T_INIT = solver.compute_equilibrium_profile(
    Z_GRID, T_surf_mean=250.0, model_id=MODEL_ID, chi=CHI)

# ── Run solver ────────────────────────────────────────────────────────────
print(f'Running {NDAYS} lunar day(s) at target site …', end=' ', flush=True)
t0 = time.time()
T_PROFILE, T_ARR = solver.solve_thermal_model(
    Z_GRID, T_INIT,
    ACTUAL_LAT, ACTUAL_LON, SLOPE, ASPECT, HORIZONS, AZ_ANGLES,
    CHI, MODEL_ID, SUNSCALE, NDAYS,
    albedo=ALBEDO, emissivity=EMISSIVITY,
)
print(f'done in {time.time()-t0:.1f} s')
print(f'  Shape : {T_PROFILE.shape[0]} snapshots × {T_PROFILE.shape[1]} depth nodes')
print(f'  T range: {T_PROFILE.min():.0f} – {T_PROFILE.max():.0f} K')

# ── Extract statistics for the final lunar day ────────────────────────────
STATS  = analysis.extract_stats(T_PROFILE, T_ARR, Z_GRID)
CYCLES = analysis.get_diurnal_cycles(
    T_PROFILE, T_ARR, Z_GRID,
    depths_m=[0.0, 0.05, 0.10, 0.35, 0.70, 1.50])

print(f'  Surface : min={STATS["T_min"][0]:.0f} K  '
      f'max={STATS["T_max"][0]:.0f} K  '
      f'mean={STATS["T_mean"][0]:.0f} K')

## 4. Run Both Density Models at Both Apollo Sites

This cell is the main data-preparation step for sections §5–9. It runs the solver with both the **Discrete** and **Hayne 2017** density models at Apollo 15 and Apollo 17, then computes borestem thermal corrections at each site.

In [ ]:
print('Running discrete + Hayne models at Apollo 15 and Apollo 17 …\n')

APOLLO_DATA     = {}   # per-site, per-model raw results
APOLLO_RESULTS  = {}   # for dual_apollo_comparison (discrete model)
COMPARE_RESULTS = {}   # for model_comparison: {site: {model_key: stats}}
COMPARE_ERRORS  = {}   # for model_comparison: {site: {model_key: errors}}

for site_name, coords in APOLLO_COORDS.items():
    lat_s   = coords['lat']
    lon_s   = coords['lon']
    t_surf  = coords['T_surf_est']  # mean surface T for equilibrium init
    print(f'── {site_name}  ({lat_s}°N, {lon_s}°E)')

    # Snap to DEM
    _row, _col, _alat, _alon, _elev, _sl, _asp = dem.extract_point(
        lat_s, lon_s, ELEV_M, PIXEL_M, MAP_RES)

    # Horizon (coarser range — Apollo sites are nearly flat mare)
    _horiz = horizon.compute_horizon_profile(
        _row, _col, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500)

    site_entry = {
        'lat': _alat, 'lon': _alon,
        'slope': _sl, 'aspect': _asp, 'horizons': _horiz,
        'disc': {}, 'hayne': {},
    }
    _compare_stats  = {}
    _compare_errors = {}

    for model_key, mid in [('disc', DISC_ID), ('hayne', HAYNE_ID)]:
        _T_init = solver.compute_equilibrium_profile(
            Z_GRID, t_surf, mid, CHI)

        t0 = time.time()
        _TP, _TA = solver.solve_thermal_model(
            Z_GRID, _T_init,
            _alat, _alon, _sl, _asp, _horiz, AZ_ANGLES,
            CHI, mid, SUNSCALE, NDAYS,
            albedo=ALBEDO, emissivity=EMISSIVITY,
        )
        _stats  = analysis.extract_stats(_TP, _TA, Z_GRID)
        _errors = analysis.compute_apollo_errors(
            _stats['T_mean'], Z_GRID, site_name)
        _cycles = analysis.get_diurnal_cycles(
            _TP, _TA, Z_GRID,
            depths_m=[0.0, 0.05, 0.10, 0.35, 0.70, 1.50])

        # Conductivity profile evaluated at T_mean — needed for borestem
        _k_prof = np.array([
            models.thermal_conductivity(
                float(_stats['T_mean'][i]), float(Z_GRID[i]), CHI, mid)
            for i in range(len(Z_GRID))
        ])

        site_entry[model_key] = {
            'T_profile': _TP, 'T_arr': _TA,
            'stats': _stats, 'cycles': _cycles,
            'errors': _errors, 'k_profile': _k_prof,
        }
        model_str = 'discrete' if model_key == 'disc' else 'hayne_exponential'
        _compare_stats[model_str]  = _stats
        _compare_errors[model_str] = _errors

        print(f'   [{model_key}] {time.time()-t0:.1f}s  '
              f'RMSE={_errors["rmse"]:.2f} K  '
              f'bias={_errors["bias"]:+.2f} K')

    APOLLO_DATA[site_name]    = site_entry
    APOLLO_RESULTS[site_name] = {
        'stats':  site_entry['disc']['stats'],
        'errors': site_entry['disc']['errors'],
    }
    COMPARE_RESULTS[site_name] = _compare_stats
    COMPARE_ERRORS[site_name]  = _compare_errors
    print()

print('✓ All Apollo model runs complete.')